<a href="https://colab.research.google.com/github/kimgayeon430/IOT/blob/main/week_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 10 실습: GNN 기초

- 공통 데이터셋: Amazon Review Dataset (Magazine Subscriptions)

In [ ]:
# ====== 공통 환경 설정 ======
# 이 노트북은 Colab/Jupyter 환경을 가정합니다.
# 필요한 패키지가 없으면 아래 설치 셀을 실행하세요.

import sys, subprocess, pkgutil
from typing import List

def _is_installed(import_name: str) -> bool:
    return pkgutil.find_loader(import_name) is not None

def pip_install(pkgs: List[str]) -> None:
    # Install packages quietly. (Colab/Jupyter에서 사용)
    print("Installing:", pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# (pip 패키지명, import 이름)
# scikit-learn은 import명이 sklearn이므로 둘을 분리해서 관리합니다.
required = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("scikit-learn", "sklearn"),
    ("tqdm", "tqdm"),
    ("matplotlib", "matplotlib"),
    ("datasets", "datasets"),  # HuggingFace datasets; 이 노트북에서는 선택 사항에 가까움
]

missing = [pkg for pkg, imp in required if not _is_installed(imp)]
if missing:
    print("누락된 패키지:", missing)
    print("아래 줄의 주석을 해제하고 실행하여 설치하세요.")
    # pip_install(missing)

# PyTorch (Week 10~12에서 사용)
if not _is_installed("torch"):
    print("torch 미설치: Week 10~12 실습에서 필요합니다.")
    print("예: pip_install(['torch'])")


/tmp/ipykernel_8864/2564396905.py:9: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  return pkgutil.find_loader(import_name) is not None


# 공통: Amazon 리뷰 데이터 로딩 & 전처리

 **실습에서 공통으로 쓰는 컬럼 표준화**
- `user_id` : 고객/리뷰어 ID
- `item_id` : 상품 ID (asin/product_id 등)
- `rating` : 1~5점
- `review_text` : 리뷰 본문
- `helpful_votes`, `total_votes` : (있다면) 리뷰 품질 가중치에 활용
- `timestamp` : (있다면) 사용자별 최신 리뷰를 test로 뽑는 split에 활용


In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import shutil
import urllib.request

SEED = 42
rng = np.random.default_rng(SEED)

# ====== 사용자 설정 =======
DATA_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Magazine_Subscriptions.jsonl.gz"
DATA_DIR = Path("./data")
LOCAL_GZ_PATH = DATA_DIR / "Magazine_Subscriptions.jsonl.gz"

N_SAMPLES = 10000          # 너무 크면 실행이 느려짐 (load할 샘플 수)
MIN_INTERACTIONS_USER = 2  # leave-one-out split 후에도 train에 최소 1개 interaction이 남도록 권장
MIN_INTERACTIONS_ITEM = 1

def download_if_needed(url: str, local_path: Path, timeout: int = 30):
    """데이터가 없으면 다운로드합니다. 네트워크 오류 시 toy 데이터 fallback이 가능하도록 예외를 올립니다."""
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists() and local_path.stat().st_size > 0:
        print(f"Using cached file: {local_path}")
        return

    tmp_path = local_path.with_suffix(local_path.suffix + ".tmp")
    print(f"Downloading dataset to: {local_path}")
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response, open(tmp_path, "wb") as f:
            shutil.copyfileobj(response, f)
        tmp_path.replace(local_path)
        print("Download complete.")
    except Exception:
        if tmp_path.exists():
            tmp_path.unlink()
        raise

def load_local_jsonl_gz(local_path: Path, n_samples: int | None = None) -> pd.DataFrame:
    # n_samples가 지정되면 전체 파일을 메모리에 올리지 않고 첫 chunk만 읽습니다.
    if n_samples is not None:
        reader = pd.read_json(
            local_path,
            lines=True,
            compression="gzip",
            chunksize=n_samples,
        )
        df = next(reader).copy()
    else:
        df = pd.read_json(
            local_path,
            lines=True,
            compression="gzip",
        )
    print(f"Loaded local gz JSONL: {local_path} / rows={len(df):,}")
    return df

def _make_toy_amazon_like(n_users=200, n_items=300, n_rows=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    users = [f"U{u:04d}" for u in range(n_users)]
    items = [f"I{i:04d}" for i in range(n_items)]

    title_pool = [
        "Great value",
        "Would buy again",
        "Just okay",
        "Not recommended",
        "Fast shipping",
    ]
    text_pool = [
        "Great quality and fast shipping.",
        "Not what I expected. The material feels cheap.",
        "Works as described. Good value for the price.",
        "Terrible experience. Would not recommend.",
        "Decent product, but packaging was damaged.",
    ]

    rows = []
    for _ in range(n_rows):
        u = rng.choice(users)
        i = rng.choice(items)
        rating = int(rng.integers(1, 6))
        title = rng.choice(title_pool)
        txt = rng.choice(text_pool)
        helpful = int(rng.integers(0, 10))
        total = helpful + int(rng.integers(0, 5))
        ts = pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(rng.integers(0, 365)))
        rows.append((u, i, rating, title, txt, helpful, total, ts))
    return pd.DataFrame(rows, columns=[
        "user_id", "item_id", "rating", "review_title", "review_text",
        "helpful_votes", "total_votes", "timestamp"
    ])

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    def pick(*cands):
        for c in cands:
            if c in df.columns:
                return c
        return None

    # Amazon Reviews 2023 raw review schema 대응
    user_col = pick("user_id", "customer_id", "reviewerID", "reviewer_id", "reviewerId")
    item_col = pick("parent_asin", "asin", "product_id", "item_id")
    rating_col = pick("rating", "star_rating", "overall", "stars")
    text_col = pick("text", "review_body", "reviewText", "review_text", "review")
    title_col = pick("title", "review_headline", "summary", "review_title")
    helpful_col = pick("helpful_vote", "helpful_votes", "helpful")
    total_col = pick("total_votes", "total")
    time_col = pick("timestamp", "review_date", "unixReviewTime", "date")

    if user_col is None or item_col is None or rating_col is None or text_col is None:
        raise ValueError(
            "필수 컬럼(user/item/rating/text)을 찾지 못했습니다. "
            f"user={user_col}, item={item_col}, rating={rating_col}, text={text_col}\n"
            f"Columns: {list(df.columns)[:50]} ..."
        )

    out = pd.DataFrame({
        "user_id": df[user_col].astype(str),
        "item_id": df[item_col].astype(str),
        "rating": pd.to_numeric(df[rating_col], errors="coerce"),
        "review_text": df[text_col].fillna("").astype(str),
    })

    out["review_title"] = df[title_col].fillna("").astype(str) if title_col is not None else ""

    # helpful / total votes
    out["helpful_votes"] = 0
    out["total_votes"] = 0

    if helpful_col is not None:
        first_helpful = df[helpful_col].dropna().iloc[0] if df[helpful_col].notna().any() else None
        if isinstance(first_helpful, (list, tuple)) and len(first_helpful) >= 2:
            out["helpful_votes"] = [x[0] if isinstance(x, (list, tuple)) and len(x) >= 1 else 0 for x in df[helpful_col]]
            out["total_votes"] = [x[1] if isinstance(x, (list, tuple)) and len(x) >= 2 else 0 for x in df[helpful_col]]
        else:
            out["helpful_votes"] = pd.to_numeric(df[helpful_col], errors="coerce").fillna(0).astype(int)
            if total_col is not None:
                out["total_votes"] = pd.to_numeric(df[total_col], errors="coerce").fillna(0).astype(int)
            else:
                # total_votes 컬럼이 없으면 helpful_votes로 대체
                out["total_votes"] = out["helpful_votes"]

    # timestamp: epoch 단위(ms/s/ns)와 문자열 날짜 모두 대응
    if time_col is not None:
        if time_col == "unixReviewTime":
            out["timestamp"] = pd.to_datetime(pd.to_numeric(df[time_col], errors="coerce"), unit="s", errors="coerce")
        else:
            s = pd.to_numeric(df[time_col], errors="coerce")
            if s.notna().any():
                med = float(s.dropna().abs().median())
                if med > 1e14:       # ns epoch
                    out["timestamp"] = pd.to_datetime(s, unit="ns", errors="coerce")
                elif med > 1e11:     # ms epoch, e.g. 1690000000000
                    out["timestamp"] = pd.to_datetime(s, unit="ms", errors="coerce")
                elif med > 1e9:      # s epoch, e.g. 1690000000
                    out["timestamp"] = pd.to_datetime(s, unit="s", errors="coerce")
                else:
                    out["timestamp"] = pd.to_datetime(df[time_col], errors="coerce")
            else:
                out["timestamp"] = pd.to_datetime(df[time_col], errors="coerce")
    else:
        out["timestamp"] = pd.NaT

    out = out.dropna(subset=["rating"])
    out["rating"] = out["rating"].clip(1, 5).astype(float)
    return out

def k_core_filter(df: pd.DataFrame, min_user: int, min_item: int, max_iters: int = 10) -> pd.DataFrame:
    df = df.copy()
    for _ in range(max_iters):
        before = len(df)
        vc_u = df["user_id"].value_counts()
        vc_i = df["item_id"].value_counts()
        df = df[df["user_id"].isin(vc_u[vc_u >= min_user].index)]
        df = df[df["item_id"].isin(vc_i[vc_i >= min_item].index)]
        after = len(df)
        if after == before:
            break
    return df.copy()

def index_ids(df: pd.DataFrame):
    users = df["user_id"].unique().tolist()
    items = df["item_id"].unique().tolist()
    user2idx = {u: i for i, u in enumerate(users)}
    item2idx = {it: i for i, it in enumerate(items)}
    df = df.copy()
    df["u"] = df["user_id"].map(user2idx).astype(int)
    df["i"] = df["item_id"].map(item2idx).astype(int)
    return df, user2idx, item2idx

def leave_one_out_split(df: pd.DataFrame, seed: int = SEED):
    df = df.copy()
    if df.empty:
        raise ValueError("leave_one_out_split에 빈 DataFrame이 들어왔습니다.")

    if df["timestamp"].notna().any():
        df = df.sort_values(["user_id", "timestamp"])
        test_idx = df.groupby("user_id").tail(1).index
    else:
        test_idx = df.groupby("user_id", group_keys=False).sample(n=1, random_state=seed).index

    test = df.loc[test_idx].copy()
    train = df.drop(index=test_idx).copy()
    return train, test

def prepare_interaction_df(raw_or_standard_df: pd.DataFrame, already_standardized: bool = False) -> pd.DataFrame:
    df = raw_or_standard_df.copy() if already_standardized else standardize_columns(raw_or_standard_df)

    # toy/로컬 파일 모두에서 optional column 누락을 방지
    for col, default in {
        "review_title": "",
        "helpful_votes": 0,
        "total_votes": 0,
    }.items():
        if col not in df.columns:
            df[col] = default
    if "timestamp" not in df.columns:
        df["timestamp"] = pd.NaT

    df["review_title"] = df["review_title"].fillna("").astype(str)
    df["review_text"] = df["review_text"].fillna("").astype(str)
    df["helpful_votes"] = pd.to_numeric(df["helpful_votes"], errors="coerce").fillna(0).astype(int)
    df["total_votes"] = pd.to_numeric(df["total_votes"], errors="coerce").fillna(0).astype(int)
    df["total_votes"] = np.maximum(df["total_votes"], df["helpful_votes"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # 중복 사용자-아이템 리뷰가 여러 개면 최근 것 우선
    if df["timestamp"].notna().any():
        df = df.sort_values("timestamp").drop_duplicates(subset=["user_id", "item_id"], keep="last")
    else:
        df = df.drop_duplicates(subset=["user_id", "item_id"], keep="last")

    return k_core_filter(df, MIN_INTERACTIONS_USER, MIN_INTERACTIONS_ITEM)

# ====== 데이터 로드 =======
try:
    download_if_needed(DATA_URL, LOCAL_GZ_PATH)
    raw_df = load_local_jsonl_gz(LOCAL_GZ_PATH, N_SAMPLES)
    df = prepare_interaction_df(raw_df, already_standardized=False)
except Exception as e:
    print("\n로컬 gz JSONL 로딩 실패 → toy 데이터로 대체합니다.")
    print("에러:", repr(e))
    df = prepare_interaction_df(_make_toy_amazon_like(), already_standardized=True)

# 실제 데이터 샘플이 너무 sparse해서 k-core 이후 비면 toy 데이터로 안전하게 진행
if df.empty or df["user_id"].nunique() == 0 or df["item_id"].nunique() == 0:
    print("\nk-core 이후 사용 가능한 interaction이 없어 toy 데이터로 대체합니다.")
    df = prepare_interaction_df(_make_toy_amazon_like(), already_standardized=True)

print("\n[After preprocessing]")
print(df.head())
print(df.shape, df.columns.tolist())
print("#Users:", df["user_id"].nunique(), "#Items:", df["item_id"].nunique())

df, user2idx, item2idx = index_ids(df)
n_users = len(user2idx)
n_items = len(item2idx)

train_df, test_df = leave_one_out_split(df)

if train_df.empty or test_df.empty:
    raise ValueError("train/test split 결과가 비었습니다. MIN_INTERACTIONS_USER 또는 N_SAMPLES를 조정하세요.")

print("\nTrain/Test")
print(train_df.shape, test_df.shape)
print(f"n_users={n_users}, n_items={n_items}")
train_df.head()


Using cached file: data/Magazine_Subscriptions.jsonl.gz
Loaded local gz JSONL: data/Magazine_Subscriptions.jsonl.gz / rows=10,000

[After preprocessing]
                           user_id     item_id  rating  \
9477  AHR7ZC2G4VF4MEJMXTZANR23OJ6Q  B00005N7PT     4.0   
9476  AHR7ZC2G4VF4MEJMXTZANR23OJ6Q  B00005N7VP     5.0   
620   AGXFIFO4VWNHPAX3VJPRQ2QJSCSQ  B00005N7P8     4.0   
619   AGXFIFO4VWNHPAX3VJPRQ2QJSCSQ  B00005N7TX     5.0   
618   AGXFIFO4VWNHPAX3VJPRQ2QJSCSQ  B00005NIOR     4.0   

                                            review_text  \
9477  Discover is a fun magazine, and a much easier ...   
9476  I can't tell you how many great science fictio...   
620   Granted, it is called 'Catholic Digest' but it...   
619   &quot;Running Times&quot; is for runners who c...   
618   Hardcore runners will read "Runner's World" be...   

                                           review_title  helpful_votes  \
9477                        The lighter side of Science             8

,user_id,item_id,rating,review_text,review_title,helpful_votes,total_votes,timestamp,u,i
5444,AE22KFYQBL52KJR5BWDLFR4IZF4A,B00005NIP7,1.0,"As of today, I am still waiting to receive Tra...",Never got the magazine,5,5,2010-01-30 19:52:11,100,163
5443,AE22KFYQBL52KJR5BWDLFR4IZF4A,B00005N7TG,5.0,I had purchase 3 different magazines on Amazon...,Subscription received,0,0,2010-01-30 19:56:07,100,92
982,AE23XZF2JZLER42SHFHB6BSE5IIA,B00006KFK1,5.0,My wife loves it.,Loves it.,0,0,2014-11-17 15:27:35,454,223
981,AE23XZF2JZLER42SHFHB6BSE5IIA,B0045FEHCS,5.0,Wife loves it!,Wife loves it!,2,2,2017-02-24 11:04:33,454,420
7260,AE25ONEGFWY6QOK2VAW6ZQK3GLQQ,B001U5SPME,5.0,I subscribe to only 1 magazine: Wired. I am an...,"Educational, entertaining far beyond tech stuff",2,2,2013-01-18 22:10:36,237,248


## 공통 유틸: RMSE / Top‑K 평가 함수

- **RMSE**: explicit rating 예측 성능
- **Top‑K Hit/Recall/NDCG**: 추천 순위 성능


In [ ]:
from math import log2

def rmse(y_true, y_pred) -> float:
    """Root Mean Squared Error. 1D/ND 배열 모두에서 안전하게 동작합니다."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"shape mismatch: y_true{y_true.shape} vs y_pred{y_pred.shape}")
    if y_true.size == 0:
        return float("nan")
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# train에서 각 유저가 본 아이템 집합 (negative sampling용)
train_user_items = train_df.groupby("u")["i"].apply(set).to_dict()
all_items = np.arange(n_items, dtype=int)

def sample_negatives_for_user(u: int, pos_i: int, n_neg: int, rng: np.random.Generator):
    """유저가 train에서 본 아이템과 현재 positive 아이템을 제외하고 negative 후보를 샘플링합니다."""
    seen = set(train_user_items.get(u, set()))
    seen.add(int(pos_i))
    if len(seen) >= n_items:
        return np.array([], dtype=int)

    mask = np.ones(n_items, dtype=bool)
    mask[list(seen)] = False
    pool = all_items[mask]
    size = min(int(n_neg), len(pool))
    return rng.choice(pool, size=size, replace=False)

def dcg(rels):
    # rels: list of 0/1
    return sum(r / log2(idx + 2) for idx, r in enumerate(rels))

def ndcg_at_k(ranked_items, ground_truth_item, k: int):
    rels = [1 if int(it) == int(ground_truth_item) else 0 for it in ranked_items[:k]]
    ideal = [1] + [0] * (k - 1)
    denom = dcg(ideal)
    return 0.0 if denom == 0 else dcg(rels) / denom

def evaluate_topk(model, test_df: pd.DataFrame, k: int = 10, n_neg: int = 100, seed: int = SEED):
    rng = np.random.default_rng(seed)
    hits, ndcgs = [], []
    for row in test_df.itertuples(index=False):
        u = int(row.u)
        pos_i = int(row.i)
        negs = sample_negatives_for_user(u, pos_i, n_neg, rng)
        if len(negs) == 0:
            continue
        candidates = np.concatenate([negs, [pos_i]]).astype(int)
        scores = np.asarray(model.predict(u, candidates), dtype=float)  # higher is better
        if scores.shape[0] != candidates.shape[0]:
            raise ValueError("model.predict는 candidates와 같은 길이의 score를 반환해야 합니다.")

        # rank descending
        ranked = candidates[np.argsort(-scores)]
        hit = 1 if pos_i in ranked[:k] else 0
        hits.append(hit)
        ndcgs.append(ndcg_at_k(ranked, pos_i, k))

    if len(hits) == 0:
        return {f"Hit@{k}": 0.0, f"NDCG@{k}": 0.0, "n_eval": 0}

    return {
        f"Hit@{k}": float(np.mean(hits)),
        f"NDCG@{k}": float(np.mean(ndcgs)),
        "n_eval": int(len(hits)),
    }

print("Evaluation utilities ready.")


Evaluation utilities ready.


## 실습 목표
1. Amazon 리뷰의 user‑item 상호작용으로 **bipartite graph**를 만든다.  
2. (간단) **GCN 스타일**의 정규화 adjacency로 embedding propagation을 구현한다.  
3. (간단) **Graph Attention(GAT)** 아이디어를 구현/비교한다.  
4. Link prediction 관점으로 **(u,i) 간선 점수**를 학습하고 Top‑K를 평가한다.



## User‑Item 그래프 만들기 (Bipartite)

- 노드: `user`, `item`
- 엣지: (user가 item에 리뷰/평점 남김)

우리는 학습 데이터(train_df)의 (u,i)로 그래프를 만들고,
정규화 adjacency $\tilde A = D^{-1/2}(A+I)D^{-1/2}$ 를 구성


In [ ]:
from scipy.sparse import coo_matrix, csr_matrix, identity, diags

# train edges
u_nodes = train_df["u"].to_numpy(dtype=int)
i_nodes = train_df["i"].to_numpy(dtype=int) + n_users  # item nodes shifted

N = n_users + n_items
# undirected edges: u->i and i->u
row = np.concatenate([u_nodes, i_nodes])
col = np.concatenate([i_nodes, u_nodes])
data = np.ones_like(row, dtype=np.float32)

A = coo_matrix((data, (row, col)), shape=(N, N)).tocsr()
print("Adjacency A:", A.shape, "| nnz:", A.nnz)

# add self-loops: dense np.eye 대신 sparse identity 사용
I = identity(N, dtype=np.float32, format="csr")
A_hat = A + I

# normalized adjacency: D^{-1/2} A_hat D^{-1/2}
deg = np.asarray(A_hat.sum(axis=1)).ravel()
deg_inv_sqrt = np.zeros_like(deg, dtype=np.float32)
mask = deg > 0
deg_inv_sqrt[mask] = np.power(deg[mask], -0.5)
D_inv_sqrt = diags(deg_inv_sqrt, offsets=0, shape=(N, N), format="csr")

A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt
A_norm = A_norm.astype(np.float32).tocsr()
print("A_norm nnz:", A_norm.nnz, "| deg stats:", np.min(deg), np.median(deg), np.max(deg))

Adjacency A: (2381, 2381) | nnz: 5178
A_norm nnz: 7559 | deg stats: 1.0 2.0 61.0


## GCN 스타일 Message Passing (학습 없이 전파만 해보기)

가장 단순한 그래프 컨볼루션은 다음처럼 이해할 수 있음

$\mathbf{E}^{(k+1)} = \tilde A \mathbf{E}^{(k)}$

- $\mathbf{E}$: 모든 노드의 임베딩 행렬  
- $\tilde A$: 정규화 adjacency  
- **의미:** 각 노드는 “자기 + 이웃”의 정보를 평균(정규화)해서 새로운 임베딩을 얻는다

> 여기서는 학습을 하기 전에, 전파만 했을 때 임베딩이 어떻게 섞이는지 감각을 잡기


In [ ]:
# 임베딩 차원
DIM = 32
rng = np.random.default_rng(SEED)
E0 = 0.01 * rng.standard_normal((N, DIM)).astype(np.float32)

# 1-step propagation
E1 = A_norm @ E0

# 한 유저를 골라 이웃 아이템을 확인
u0 = int(train_df["u"].iloc[0])
neighbors = A[u0].nonzero()[1]  # indices in combined graph
neighbor_items = [int(n - n_users) for n in neighbors if n >= n_users][:10]

print("Example user u0=", u0)
print("Neighbor items (first 10):", neighbor_items)

# 직관 체크: E1[u0]는 (u0 + 이웃)의 평균이 섞인 것
print("E0[u0][:5] =", np.asarray(E0[u0]).ravel()[:5])
print("E1[u0][:5] =", np.asarray(E1[u0]).ravel()[:5])


Example user u0= 100
Neighbor items (first 10): [92, 163]
E0[u0][:5] = [-0.0057367   0.01163496 -0.00306202  0.01104495  0.00428965]
E1[u0][:5] = [-0.00242663 -0.00257468 -0.00489169  0.00741596  0.00595804]


## GCN‑style 임베딩으로 Link prediction 학습 (BPR)

이제 임베딩을 학습합니다.  
- 노드 임베딩을 초기화하고
- $\tilde A$로 K번 전파한 임베딩을 사용해
- 관측된 (u, i⁺) 간선 점수는 높이고, 미관측 (u, i⁻) 점수는 낮추도록(BPR) 학습

### TODO
- 전파 횟수 K를 0/1/2로 바꾸면 성능이 어떻게 변하는가?


In [ ]:
# ====== PyTorch 기반 GCN-style BPR 학습 ======
try:
    import torch
    import torch.nn as nn
except Exception as e:
    raise ImportError("torch가 필요합니다. setup 셀에서 torch 설치 후 다시 실행하세요.") from e

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# scipy sparse -> torch sparse
A_norm_coo = A_norm.tocoo()
indices = torch.tensor(np.vstack([A_norm_coo.row, A_norm_coo.col]), dtype=torch.long)
values = torch.tensor(A_norm_coo.data, dtype=torch.float32)
A_norm_t = torch.sparse_coo_tensor(indices, values, size=A_norm_coo.shape).coalesce().to(device)

# positive edges for sampling (implicit: train interaction 모두 positive)
pos_by_user_all = train_df.groupby("u")["i"].apply(list).to_dict()
users_with_pos = np.array(list(pos_by_user_all.keys()), dtype=int)

if len(users_with_pos) == 0:
    raise ValueError("BPR 학습에 사용할 positive interaction이 없습니다.")

def sample_one_negative(u: int, rng: np.random.Generator):
    seen = set(train_user_items.get(int(u), set()))
    if len(seen) >= n_items:
        return None

    # 대부분 sparse하므로 빠른 rejection sampling을 먼저 시도
    for _ in range(100):
        neg_i = int(rng.integers(0, n_items))
        if neg_i not in seen:
            return neg_i

    # 유저가 너무 많은 아이템을 본 예외 상황에서도 무한 루프를 피함
    mask = np.ones(n_items, dtype=bool)
    mask[list(seen)] = False
    candidates = all_items[mask]
    if len(candidates) == 0:
        return None
    return int(rng.choice(candidates))

def sample_bpr_batch(batch_size: int, rng: np.random.Generator):
    us, pos_is, neg_is = [], [], []
    max_attempts = max(batch_size * 10, 1000)
    attempts = 0

    while len(us) < batch_size and attempts < max_attempts:
        attempts += 1
        u = int(rng.choice(users_with_pos))
        pos_i = int(rng.choice(pos_by_user_all[u]))
        neg_i = sample_one_negative(u, rng)
        if neg_i is None:
            continue
        us.append(u)
        pos_is.append(pos_i)
        neg_is.append(neg_i)

    if len(us) == 0:
        raise ValueError("negative sample을 만들 수 없습니다. n_items 또는 train split을 확인하세요.")

    return np.array(us, dtype=int), np.array(pos_is, dtype=int), np.array(neg_is, dtype=int)

class GCNPropBPR(nn.Module):
    def __init__(self, n_users: int, n_items: int, dim: int = 64, K: int = 1):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.N = n_users + n_items
        self.dim = dim
        self.K = K
        self.E0 = nn.Embedding(self.N, dim)
        nn.init.normal_(self.E0.weight, std=0.01)

    def propagate(self, A_norm_t: torch.Tensor):
        E = self.E0.weight
        for _ in range(self.K):
            E = torch.sparse.mm(A_norm_t, E)
        return E

    def score(self, E: torch.Tensor, u_idx: torch.Tensor, i_idx: torch.Tensor):
        # user node: u, item node: n_users + i
        u_emb = E[u_idx]
        i_emb = E[self.n_users + i_idx]
        return (u_emb * i_emb).sum(dim=1)

def train_gcn_bpr(K=1, dim=64, lr=1e-3, reg=1e-4, epochs=5, steps_per_epoch=200, batch_size=1024, seed=SEED):
    rng = np.random.default_rng(seed)
    model = GCNPropBPR(n_users, n_items, dim=dim, K=K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_steps = 0
        for _ in range(steps_per_epoch):
            us, pos_is, neg_is = sample_bpr_batch(batch_size, rng)
            actual_batch_size = len(us)
            us = torch.tensor(us, dtype=torch.long, device=device)
            pos_is = torch.tensor(pos_is, dtype=torch.long, device=device)
            neg_is = torch.tensor(neg_is, dtype=torch.long, device=device)

            E = model.propagate(A_norm_t)
            s_pos = model.score(E, us, pos_is)
            s_neg = model.score(E, us, neg_is)
            loss = -torch.nn.functional.logsigmoid(s_pos - s_neg).mean()
            # L2 reg on embeddings (simple)
            l2 = (
                model.E0.weight[us].pow(2).sum()
                + model.E0.weight[n_users + pos_is].pow(2).sum()
                + model.E0.weight[n_users + neg_is].pow(2).sum()
            ) / actual_batch_size
            loss = loss + reg * l2

            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += float(loss.item())
            total_steps += 1

        print(f"Epoch {ep:02d}/{epochs} | loss={total_loss/max(total_steps, 1):.4f}")

    return model

# 빠른 실습 실행을 위한 기본값입니다. 더 긴 학습은 epochs/steps_per_epoch를 늘려보세요.
gcn_model = train_gcn_bpr(K=1, dim=64, epochs=2, steps_per_epoch=10, batch_size=512)

# 평가를 위해 model.predict 래퍼 작성
class TorchGCNWrapper:
    def __init__(self, model: GCNPropBPR, A_norm_t: torch.Tensor):
        self.model = model
        self.A = A_norm_t
        self.E_cached = None

    def _get_E(self):
        if self.E_cached is None:
            self.model.eval()
            with torch.no_grad():
                self.E_cached = self.model.propagate(self.A).detach().cpu().numpy()
        return self.E_cached

    def predict(self, u: int, items):
        E = self._get_E()
        items = np.asarray(items, dtype=int)
        u_emb = E[int(u)]
        i_emb = E[n_users + items]
        return (u_emb[None, :] * i_emb).sum(axis=1)

gcn_wrap = TorchGCNWrapper(gcn_model, A_norm_t)
print("Top‑K metrics (GCN-style BPR):", evaluate_topk(gcn_wrap, test_df, k=10, n_neg=50))


device: cpu
Epoch 01/2 | loss=0.6918
Epoch 02/2 | loss=0.6886
Top‑K metrics (GCN-style BPR): {'Hit@10': 0.36911764705882355, 'NDCG@10': 0.22901279195271554, 'n_eval': 1360}


## Graph Attention(GAT) 스타일 Link Prediction 학습 (BPR)

- **GCN**: $D^{-1/2}(A+I)D^{-1/2}$ 로 고정된 정규화 가중치를 사용
- **Graph Attention**: 각 노드가 이웃별 중요도를 학습해서 weighted sum으로 메시지를 집계
- 학습 목적: BPR loss로 관측된 $(u, i^+)$ 점수를 미관측 $(u, i^-)$보다 크게 만들기

In [ ]:
# ====== PyTorch 기반 Graph Attention(GAT) BPR 학습 ======
# GCN 셀에서 정의한 device, sample_bpr_batch, evaluate_topk 등을 그대로 사용합니다.
import torch.nn.functional as F

# GAT edge index 만들기
# - A는 user-item bipartite graph의 undirected adjacency입니다.
# - GAT에서는 자기 자신도 볼 수 있도록 self-loop를 추가합니다.
A_gat = (A + identity(N, dtype=np.float32, format="csr")).tocoo()
edge_index = torch.tensor(
    np.vstack([A_gat.row, A_gat.col]),
    dtype=torch.long,
    device=device,
)
print("GAT edge_index:", tuple(edge_index.shape), "| edges:", edge_index.shape[1])


class GATLayer(nn.Module):
    """Sparse adjacency를 사용하는 single-head GAT layer."""
    def __init__(self, dim: int, dropout: float = 0.0, negative_slope: float = 0.2):
        super().__init__()
        self.lin = nn.Linear(dim, dim, bias=False)
        self.attn_src = nn.Parameter(torch.empty(dim))
        self.attn_dst = nn.Parameter(torch.empty(dim))
        self.dropout = dropout
        self.negative_slope = negative_slope
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.attn_src.view(1, -1))
        nn.init.xavier_uniform_(self.attn_dst.view(1, -1))

    def forward(self, E: torch.Tensor, edge_index: torch.Tensor):
        # row(dst): 정보를 받는 노드, col(src): 메시지를 보내는 노드
        row, col = edge_index
        H = self.lin(E)

        ################################################
        #           To Do (attention 구현)              #
        # e = LeakyReLU(a_dst^T H_dst + a_src^T H_src) #
        ################################################

        e = F.leaky_relu(
          (H[row] * self.attn_dst).sum(dim=1) + (H[col] * self.attn_src).sum(dim=1),
          negative_slope=self.negative_slope
        )

        # sparse softmax: 같은 dst(row)를 가진 이웃들끼리 softmax
        attn_sparse = torch.sparse_coo_tensor(edge_index, e, size=(E.size(0), E.size(0))).coalesce()
        alpha_sparse = torch.sparse.softmax(attn_sparse, dim=1)
        alpha = alpha_sparse.values()

        if self.dropout > 0:
            alpha = F.dropout(alpha, p=self.dropout, training=self.training)
            alpha_sparse = torch.sparse_coo_tensor(
                alpha_sparse.indices(), alpha, size=alpha_sparse.shape
            ).coalesce()

        # h'_v = sum_{u in N(v)} alpha_{vu} W h_u
        out = torch.sparse.mm(alpha_sparse, H)
        return out


class GATBPR(nn.Module):
    """User-item bipartite graph용 Graph Attention + BPR 모델."""
    def __init__(self, n_users: int, n_items: int, dim: int = 64, K: int = 1, dropout: float = 0.0):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.N = n_users + n_items
        self.dim = dim
        self.K = K
        self.dropout = dropout

        self.E0 = nn.Embedding(self.N, dim)
        nn.init.normal_(self.E0.weight, std=0.01)
        self.layers = nn.ModuleList([GATLayer(dim, dropout=dropout) for _ in range(K)])

    def propagate(self, edge_index: torch.Tensor):
        E = self.E0.weight
        for layer in self.layers:
            E = layer(E, edge_index)
            E = F.elu(E)
            E = F.normalize(E, p=2, dim=1)
        return E

    def score(self, E: torch.Tensor, u_idx: torch.Tensor, i_idx: torch.Tensor):
        u_emb = E[u_idx]
        i_emb = E[self.n_users + i_idx]
        return (u_emb * i_emb).sum(dim=1)


def train_gat_bpr(K=1, dim=64, lr=1e-3, reg=1e-4, epochs=5, steps_per_epoch=200, batch_size=1024, dropout=0.0, seed=SEED):
    rng = np.random.default_rng(seed)
    model = GATBPR(n_users, n_items, dim=dim, K=K, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_steps = 0

        for _ in range(steps_per_epoch):
            us, pos_is, neg_is = sample_bpr_batch(batch_size, rng)
            actual_batch_size = len(us)

            us = torch.tensor(us, dtype=torch.long, device=device)
            pos_is = torch.tensor(pos_is, dtype=torch.long, device=device)
            neg_is = torch.tensor(neg_is, dtype=torch.long, device=device)

            E = model.propagate(edge_index)
            s_pos = model.score(E, us, pos_is)
            s_neg = model.score(E, us, neg_is)

            # BPR loss: positive score가 negative score보다 커지도록 학습
            loss = -F.logsigmoid(s_pos - s_neg).mean()

            # sampled node embedding L2 regularization
            l2 = (
                model.E0.weight[us].pow(2).sum()
                + model.E0.weight[n_users + pos_is].pow(2).sum()
                + model.E0.weight[n_users + neg_is].pow(2).sum()
            ) / actual_batch_size
            loss = loss + reg * l2

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += float(loss.item())
            total_steps += 1

        print(f"Epoch {ep:02d}/{epochs} | loss={total_loss/max(total_steps, 1):.4f}")

    return model


# 빠른 실습 실행 기본값입니다. 성능 비교는 epochs/steps_per_epoch를 늘려보세요.
gat_model = train_gat_bpr(K=1, dim=64, epochs=2, steps_per_epoch=10, batch_size=512, dropout=0.0)


class TorchGATWrapper:
    def __init__(self, model: GATBPR, edge_index: torch.Tensor):
        self.model = model
        self.edge_index = edge_index
        self.E_cached = None

    def _get_E(self):
        if self.E_cached is None:
            self.model.eval()
            with torch.no_grad():
                self.E_cached = self.model.propagate(self.edge_index).detach().cpu().numpy()
        return self.E_cached

    def predict(self, u: int, items):
        E = self._get_E()
        items = np.asarray(items, dtype=int)
        u_emb = E[int(u)]
        i_emb = E[n_users + items]
        return (u_emb[None, :] * i_emb).sum(axis=1)


gat_wrap = TorchGATWrapper(gat_model, edge_index)
print("Top‑K metrics (Graph Attention BPR):", evaluate_topk(gat_wrap, test_df, k=10, n_neg=50))


GAT edge_index: (2, 7559) | edges: 7559
Epoch 01/2 | loss=0.4011
Epoch 02/2 | loss=0.3589
Top‑K metrics (Graph Attention BPR): {'Hit@10': 0.2713235294117647, 'NDCG@10': 0.1306830976671437, 'n_eval': 1360}
